# NB_SEC_Lifecycle · Phase 5 · moves requests through their lifecycle and queues mail
Order in PL_SEC_Lifecycle: **Lifecycle** → Sync → Notify. Implements POL-02 (end dates), POL-08 (leavers), extensions and removals.

In [ ]:
%run NB_SEC_Common

In [ ]:
r = sql_df("""SELECT r.RequestId, r.RequestNumber, r.RequestType, r.UserId, r.RoleTemplateId, r.ScopeValue, r.StartDate, r.EndDate, r.Status,
                     u.UPN, u.ManagerUPN, u.EmploymentStatus
              FROM sec.AccessRequest r JOIN sec.UserProfile u ON u.UserId = r.UserId""")
r["StartDate"] = pd.to_datetime(r["StartDate"]).dt.date; r["EndDate"] = pd.to_datetime(r["EndDate"]).dt.date
changes = []
def set_status(row, status, reason, action):
    sql_exec("UPDATE sec.AccessRequest SET Status = ?, StatusReason = ? WHERE RequestId = ?", (status, reason, int(row.RequestId)))
    audit(action, f"{row.UPN} {row.RequestNumber}", request_id=int(row.RequestId), before={"status": row.Status}, after={"status": status, "reason": reason})
    changes.append((row.RequestNumber, row.Status, status, reason))

In [ ]:
# 1. Leavers lose everything (POL-08)
for row in r[(r.EmploymentStatus == "Leaver") & r.Status.isin(["Pending", "Approved", "Active"])].itertuples():
    set_status(row, "Revoked", "Leaver", "REQUEST_REVOKED")
    queue_mail(int(row.RequestId), "REVOKED", row.ManagerUPN, CONFIG["security_team_mail"])

# 1b. Requests that break a role rule are rejected: MaxDays, scope type, role conflicts (sec.vw_RequestViolations)
bad = sql_df("SELECT RequestId, RuleBroken, Detail FROM sec.vw_RequestViolations")
for v in bad.drop_duplicates("RequestId").itertuples():
    for row in r[(r.RequestId == v.RequestId) & r.Status.isin(["Pending", "Approved"])].itertuples():
        set_status(row, "Rejected", f"{v.RuleBroken}: {v.Detail}"[:200], "REQUEST_REJECTED")
        queue_mail(int(row.RequestId), "REJECTED", row.UPN, row.ManagerUPN)
r.loc[r.RequestId.isin(bad.RequestId), "Status"] = "Rejected"

# 2. Extensions and removals that were approved
active = r[r.Status == "Active"]
for row in r[(r.Status == "Approved") & r.RequestType.isin(["Extend", "Remove"])].itertuples():
    match = active[(active.UserId == row.UserId) & (active.RoleTemplateId == row.RoleTemplateId) & (active.ScopeValue == row.ScopeValue)]
    for m in match.itertuples():
        reason = f"Superseded by {row.RequestNumber}" if row.RequestType == "Extend" else f"Removed by {row.RequestNumber}"
        set_status(m, "Revoked" if row.RequestType == "Remove" else "Expired", reason, "REQUEST_" + row.RequestType.upper() + "ED")
    if row.RequestType == "Extend":
        set_status(row, "Active", "Extension applied", "REQUEST_EXTENDED"); queue_mail(int(row.RequestId), "EXTENDED", row.UPN, row.ManagerUPN)
    else:
        set_status(row, "Expired", "Removal completed", "REQUEST_REMOVED"); queue_mail(int(row.RequestId), "REVOKED", row.UPN, row.ManagerUPN)

# 3. Approved access whose start date has arrived becomes Active (mail is sent after Sync applies it)
for row in r[(r.Status == "Approved") & (r.RequestType == "New") & (r.StartDate <= TODAY) & (r.EndDate >= TODAY) & (r.EmploymentStatus == "Active")].itertuples():
    set_status(row, "Active", "Start date reached", "REQUEST_ACTIVATED")
    queue_mail(int(row.RequestId), "ENABLED", row.UPN, row.ManagerUPN)

# 4. Active access past its end date is removed (POL-02)
for row in r[(r.Status == "Active") & (r.EndDate < TODAY)].itertuples():
    set_status(row, "Expired", "End date reached", "REQUEST_EXPIRED")
    queue_mail(int(row.RequestId), "REVOKED", row.UPN, row.ManagerUPN)

# 5. Expiry warnings at T-7 and T-1
r2 = sql_df("""SELECT a.RequestId, u.UPN, u.ManagerUPN, DATEDIFF(DAY, CAST(GETDATE() AS DATE), a.EndDate) DaysLeft
               FROM sec.AccessRequest a JOIN sec.UserProfile u ON u.UserId = a.UserId
               WHERE a.Status = 'Active' AND DATEDIFF(DAY, CAST(GETDATE() AS DATE), a.EndDate) IN (1, 7)""")
warned = 0
for row in r2.itertuples():
    warned += queue_mail(int(row.RequestId), "EXPIRY_T1" if row.DaysLeft == 1 else "EXPIRY_T7", row.UPN, row.ManagerUPN)

print(f"{len(changes)} status changes, {warned} expiry warnings queued")
display(pd.DataFrame(changes, columns=["request", "from", "to", "reason"]))